# Module 11 — Introduction to Graphs

This is the worked reference notebook: run live in lecture, fully solved.
The version students receive with TODOs in place of the solved parts is
`assignments/pds/a10-graphs/starter/graphs.py`.

## 1. Graph representations (Lecture 1)

In [1]:
class GraphMatrix:
    def __init__(self, n):
        self.n = n
        self.matrix = [[0] * n for _ in range(n)]
    def add_edge(self, u, v, weight=1, directed=False):
        self.matrix[u][v] = weight
        if not directed:
            self.matrix[v][u] = weight
    def has_edge(self, u, v):
        return self.matrix[u][v] != 0

class GraphList:
    def __init__(self, n):
        self.n = n
        self.adj = [[] for _ in range(n)]
    def add_edge(self, u, v, weight=1, directed=False):
        self.adj[u].append((v, weight))
        if not directed:
            self.adj[v].append((u, weight))
    def has_edge(self, u, v):
        return any(neighbor == v for neighbor, weight in self.adj[u])

edges = [(0,1),(0,2),(1,2),(1,3),(3,4)]
gm = GraphMatrix(5)
gl = GraphList(5)
for u, v in edges:
    gm.add_edge(u, v)
    gl.add_edge(u, v)

for u in range(5):
    for v in range(5):
        assert gm.has_edge(u, v) == gl.has_edge(u, v)

nonzero_cells = sum(1 for row in gm.matrix for x in row if x != 0)
total_list_entries = sum(len(lst) for lst in gl.adj)
assert nonzero_cells == total_list_entries == 10
print("Graph representation checks passed, matching Lecture 1 exactly (10 = 10)")

Graph representation checks passed, matching Lecture 1 exactly (10 = 10)


## 2. BFS and DFS (Lecture 2)

In [2]:
from collections import deque

adj = {0:[1,2], 1:[0,2,3], 2:[0,1], 3:[1,4], 4:[3]}

def bfs(adj, start):
    visited = {start}
    queue = deque([start])
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in adj[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return order

def dfs(adj, start):
    visited = set()
    stack = [start]
    order = []
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        order.append(node)
        for neighbor in adj[node]:
            if neighbor not in visited:
                stack.append(neighbor)
    return order

assert bfs(adj, 0) == [0, 1, 2, 3, 4]
assert dfs(adj, 0) == [0, 2, 1, 3, 4]
assert bfs(adj, 4) == [4, 3, 1, 0, 2]
assert dfs(adj, 4) == [4, 3, 1, 2, 0]
print("BFS and DFS checks passed, matching Lecture 2's exact traces")

BFS and DFS checks passed, matching Lecture 2's exact traces


## 3. Connected components, cycle detection, topological sort (Lecture 3)

In [3]:
def connected_components(adj, n):
    visited = set()
    components = []
    for node in range(n):
        if node not in visited:
            comp = bfs(adj, node)
            visited.update(comp)
            components.append(comp)
    return components

islands_adj = {0:[1],1:[0],2:[3],3:[2,4],4:[3],5:[]}
assert connected_components(islands_adj, 6) == [[0,1],[2,3,4],[5]]

def has_cycle_undirected(adj, n):
    visited = set()
    def dfs_check(node, parent):
        visited.add(node)
        for neighbor in adj[node]:
            if neighbor not in visited:
                if dfs_check(neighbor, node):
                    return True
            elif neighbor != parent:
                return True
        return False
    for node in range(n):
        if node not in visited:
            if dfs_check(node, -1):
                return True
    return False

triangle = {0:[1,2],1:[0,2],2:[0,1]}
assert has_cycle_undirected(triangle, 3) == True

def has_cycle_directed(adj, n):
    WHITE, GRAY, BLACK = 0, 1, 2
    color = [WHITE] * n
    def dfs_check(node):
        color[node] = GRAY
        for neighbor in adj[node]:
            if color[neighbor] == GRAY:
                return True
            if color[neighbor] == WHITE and dfs_check(neighbor):
                return True
        color[node] = BLACK
        return False
    return any(dfs_check(node) for node in range(n) if color[node] == WHITE)

directed_cycle = {0:[1],1:[2],2:[0]}
assert has_cycle_directed(directed_cycle, 3) == True

course_adj = {0:[1,2], 1:[3], 2:[3], 3:[]}
assert has_cycle_directed(course_adj, 4) == False   # confirmed a genuine DAG

def topological_sort(adj, n):
    visited = set()
    order = []
    def dfs_visit(node):
        visited.add(node)
        for neighbor in adj[node]:
            if neighbor not in visited:
                dfs_visit(neighbor)
        order.append(node)
    for node in range(n):
        if node not in visited:
            dfs_visit(node)
    return order[::-1]

assert topological_sort(course_adj, 4) == [0, 2, 1, 3]
print("Connected components, cycle detection, and topological sort checks passed, matching Lecture 3")

Connected components, cycle detection, and topological sort checks passed, matching Lecture 3


## 4. Kahn's algorithm (Lecture 3 MTech)

In [4]:
def kahn_topological_sort(adj, n):
    in_degree = [0] * n
    for u in range(n):
        for v in adj[u]:
            in_degree[v] += 1
    queue = deque([node for node in range(n) if in_degree[node] == 0])
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in adj[node]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)
    return order if len(order) == n else None

assert kahn_topological_sort(course_adj, 4) == [0, 1, 2, 3]
assert kahn_topological_sort(directed_cycle, 3) is None   # cycle correctly detected
print("Kahn's algorithm checks passed")

Kahn's algorithm checks passed


## 5. Weighted-graph BFS failure, and negative-weight Dijkstra failure (Lecture 4)

In [5]:
import heapq

def dijkstra(edges, start, n):
    dist = {start: 0}
    visited = set()
    pq = [(0, start)]
    while pq:
        d, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        for v, w in edges.get(u, []):
            if v in visited:
                continue
            nd = d + w
            if nd < dist.get(v, float('inf')):
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return dist

def bfs_hops(adj, start):
    dist = {start: 0}
    q = deque([start])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

weighted_edges = {0:[(1,10),(2,1)], 1:[(0,10),(2,1)], 2:[(0,1),(1,1)]}
weighted_adj = {0:[1,2], 1:[0,2], 2:[0,1]}

true_dist = dijkstra(weighted_edges, 0, 3)
hop_dist = bfs_hops(weighted_adj, 0)
assert true_dist[1] == 2      # true shortest via 0->2->1
assert hop_dist[1] == 1        # BFS wrongly "prefers" the direct edge by hop count
print("Weighted BFS-failure check passed: true dist=2, BFS hop count=1 (matches Lecture 4)")

# Negative-weight Dijkstra failure
neg_edges = {0:[(1,1),(2,4)], 1:[], 2:[(1,-10)]}
neg_dist = dijkstra(neg_edges, 0, 3)
assert neg_dist[1] == 1   # WRONG answer, reported due to early finalization
true_neg_shortest = 4 + (-10)
assert true_neg_shortest == -6   # the actual shortest path, which Dijkstra misses
print("Negative-weight Dijkstra failure check passed: reports 1, true shortest is -6 (matches Lecture 4)")

Weighted BFS-failure check passed: true dist=2, BFS hop count=1 (matches Lecture 4)
Negative-weight Dijkstra failure check passed: reports 1, true shortest is -6 (matches Lecture 4)
